In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

In [ ]:
import lightning as L
import torch

from arch.vae.vae import VAE
from utils.const import SEED
from utils.utils import setup_device

model_path = "logs/vae_acdc/marginal-kl/checkpoints/epoch=47-step=5136.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load model
model = VAE.load_from_checkpoint(model_path)

In [ ]:
lerp_dim = 6

num_samples = 6
num_lerp = 10

# Sample from latent space
z = torch.randn(num_samples, model.hparams.latent_dim).to(device)

# Duplicate samples
z = z.unsqueeze(1).repeat(1, num_lerp, 1)

# Replace chosen dimension with lerped values
# -2 to 2 covers 98 percentile of standard Normal distribution
lerp_values = torch.linspace(-2, 2, num_lerp).to(device)
z[:, :, lerp_dim] = lerp_values.unsqueeze(0).repeat(num_samples, 1)

# Stack samples
z = z.view(-1, model.hparams.latent_dim)

print(z.shape)
print(z[:num_lerp])

In [ ]:
from utils.utils import show_samples

with torch.no_grad():
    model.eval()
    model.to(device)
    
    # Generate segmentation maps from latent variables
    x_fake: torch.Tensor = model.decoder(z)

# View generations
generations = torch.argmax(x_fake, dim=1).unsqueeze(1)
show_samples(generations, rgb=False, ncol=num_lerp, figsize=(num_lerp, num_samples))